In [2]:
import pandas as pd
import numpy as np
import re
import holidays
import os

In [8]:
subway_2023 = pd.read_csv('서울교통공사_역별 시간대별 승하차인원(23.1~23.12).csv', encoding='cp949')
display(subway_2023.head())

subway_2024= pd.read_csv('서울교통공사_역별 시간대별 승하차인원(24.1~24.12).csv', encoding='cp949')
display(subway_2024.head())

,연번,수송일자,호선,역번호,역명,승하차구분,06시이전,06-07시간대,07-08시간대,08-09시간대,...,15-16시간대,16-17시간대,17-18시간대,18-19시간대,19-20시간대,20-21시간대,21-22시간대,22-23시간대,23-24시간대,24시이후
0,1,2023-01-01,1호선,150,서울역,승차,215,145,231,594,...,2655,2509,2696,2549,2462,2177,2190,1808,734,7
1,2,2023-01-01,1호선,150,서울역,하차,154,636,595,939,...,2282,2295,2526,1930,1897,1487,991,609,280,46
2,3,2023-01-01,1호선,151,시청,승차,48,73,106,194,...,843,895,959,985,670,630,515,330,146,0
3,4,2023-01-01,1호선,151,시청,하차,64,247,293,463,...,602,575,533,456,285,267,246,154,79,18
4,5,2023-01-01,1호선,152,종각,승차,407,235,158,201,...,1145,1402,1223,1272,911,913,906,602,232,3


,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2024-01-01,1호선,150,서울역,승차,383,257,308,975,...,2716,2882,2871,2685,2922,2031,2279,1729,868,43
1,2,2024-01-01,1호선,150,서울역,하차,249,867,834,1201,...,2615,2501,2829,2095,1833,1465,1031,585,298,82
2,3,2024-01-01,1호선,151,시청,승차,188,92,167,245,...,935,1003,978,1116,951,909,723,462,176,3
3,4,2024-01-01,1호선,151,시청,하차,103,276,292,451,...,834,697,670,564,351,302,272,120,89,38
4,5,2024-01-01,1호선,152,종각,승차,970,374,193,205,...,1383,1578,1435,1422,1287,1318,1106,775,274,8


In [14]:
subway_2023

,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2023-01-01,1호선,150,서울역,승차,215,145,231,594,...,2655,2509,2696,2549,2462,2177,2190,1808,734,7
1,2,2023-01-01,1호선,150,서울역,하차,154,636,595,939,...,2282,2295,2526,1930,1897,1487,991,609,280,46
2,3,2023-01-01,1호선,151,시청,승차,48,73,106,194,...,843,895,959,985,670,630,515,330,146,0
3,4,2023-01-01,1호선,151,시청,하차,64,247,293,463,...,602,575,533,456,285,267,246,154,79,18
4,5,2023-01-01,1호선,152,종각,승차,407,235,158,201,...,1145,1402,1223,1272,911,913,906,602,232,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199265,199266,2023-12-31,8호선,2826,수진,하차,5,45,97,72,...,192,191,197,222,232,191,171,183,134,73
199266,199267,2023-12-31,8호선,2827,모란,승차,36,68,65,94,...,206,209,210,187,144,130,130,145,93,8
199267,199268,2023-12-31,8호선,2827,모란,하차,14,72,50,49,...,182,213,190,235,185,190,202,199,106,262
199268,199269,2023-12-31,8호선,2828,남위례,승차,20,52,56,108,...,292,309,260,209,121,135,228,178,77,17


In [15]:
subway_2024

,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2024-01-01,1호선,150,서울역,승차,383,257,308,975,...,2716,2882,2871,2685,2922,2031,2279,1729,868,43
1,2,2024-01-01,1호선,150,서울역,하차,249,867,834,1201,...,2615,2501,2829,2095,1833,1465,1031,585,298,82
2,3,2024-01-01,1호선,151,시청,승차,188,92,167,245,...,935,1003,978,1116,951,909,723,462,176,3
3,4,2024-01-01,1호선,151,시청,하차,103,276,292,451,...,834,697,670,564,351,302,272,120,89,38
4,5,2024-01-01,1호선,152,종각,승차,970,374,193,205,...,1383,1578,1435,1422,1287,1318,1106,775,274,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199405,199406,2024-12-31,8호선,2826,수진,하차,14,85,152,477,...,355,414,465,468,376,261,248,246,175,68
199406,199407,2024-12-31,8호선,2827,모란,승차,80,103,332,400,...,389,537,440,381,247,136,170,140,90,50
199407,199408,2024-12-31,8호선,2827,모란,하차,19,109,138,428,...,263,340,340,374,281,193,212,207,141,128
199408,199409,2024-12-31,8호선,2828,남위례,승차,44,236,686,811,...,656,469,556,513,285,245,216,213,75,46


In [9]:
subway_2023 = subway_2023.rename(columns={
    '수송일자': '날짜',
    '승하차구분': '구분',
    '06시이전': '06시 이전',
    '06-07시간대': '06시-07시',
    '07-08시간대': '07시-08시',
    '08-09시간대': '08시-09시',
    '09-10시간대': '09시-10시',
    '10-11시간대': '10시-11시',
    '11-12시간대': '11시-12시',
    '12-13시간대': '12시-13시',
    '13-14시간대': '13시-14시',
    '14-15시간대': '14시-15시',
    '15-16시간대': '15시-16시',
    '16-17시간대': '16시-17시',
    '17-18시간대': '17시-18시',
    '18-19시간대': '18시-19시',
    '19-20시간대': '19시-20시',
    '20-21시간대': '20시-21시',
    '21-22시간대': '21시-22시',
    '22-23시간대': '22시-23시',
    '23-24시간대': '23시-24시',
    '24시이후': '24시 이후'
})

In [10]:
subway_2023.head()

,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2023-01-01,1호선,150,서울역,승차,215,145,231,594,...,2655,2509,2696,2549,2462,2177,2190,1808,734,7
1,2,2023-01-01,1호선,150,서울역,하차,154,636,595,939,...,2282,2295,2526,1930,1897,1487,991,609,280,46
2,3,2023-01-01,1호선,151,시청,승차,48,73,106,194,...,843,895,959,985,670,630,515,330,146,0
3,4,2023-01-01,1호선,151,시청,하차,64,247,293,463,...,602,575,533,456,285,267,246,154,79,18
4,5,2023-01-01,1호선,152,종각,승차,407,235,158,201,...,1145,1402,1223,1272,911,913,906,602,232,3


In [11]:
subway = pd.concat(
    [subway_2023, subway_2024],
    ignore_index=True
)

In [12]:
subway.head()

,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2023-01-01,1호선,150,서울역,승차,215,145,231,594,...,2655,2509,2696,2549,2462,2177,2190,1808,734,7
1,2,2023-01-01,1호선,150,서울역,하차,154,636,595,939,...,2282,2295,2526,1930,1897,1487,991,609,280,46
2,3,2023-01-01,1호선,151,시청,승차,48,73,106,194,...,843,895,959,985,670,630,515,330,146,0
3,4,2023-01-01,1호선,151,시청,하차,64,247,293,463,...,602,575,533,456,285,267,246,154,79,18
4,5,2023-01-01,1호선,152,종각,승차,407,235,158,201,...,1145,1402,1223,1272,911,913,906,602,232,3


In [13]:
subway

,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2023-01-01,1호선,150,서울역,승차,215,145,231,594,...,2655,2509,2696,2549,2462,2177,2190,1808,734,7
1,2,2023-01-01,1호선,150,서울역,하차,154,636,595,939,...,2282,2295,2526,1930,1897,1487,991,609,280,46
2,3,2023-01-01,1호선,151,시청,승차,48,73,106,194,...,843,895,959,985,670,630,515,330,146,0
3,4,2023-01-01,1호선,151,시청,하차,64,247,293,463,...,602,575,533,456,285,267,246,154,79,18
4,5,2023-01-01,1호선,152,종각,승차,407,235,158,201,...,1145,1402,1223,1272,911,913,906,602,232,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
398675,199406,2024-12-31,8호선,2826,수진,하차,14,85,152,477,...,355,414,465,468,376,261,248,246,175,68
398676,199407,2024-12-31,8호선,2827,모란,승차,80,103,332,400,...,389,537,440,381,247,136,170,140,90,50
398677,199408,2024-12-31,8호선,2827,모란,하차,19,109,138,428,...,263,340,340,374,281,193,212,207,141,128
398678,199409,2024-12-31,8호선,2828,남위례,승차,44,236,686,811,...,656,469,556,513,285,245,216,213,75,46


In [16]:
subway.to_csv(
    'subway_2023-2024.csv',
    index=False,
    encoding='utf-8-sig'
)

In [3]:
weather_2023 = pd.read_csv('20230101-20231231 날씨.csv', encoding='cp949')
display(weather_2023.head())

weather_2024 = pd.read_csv('20240101-20241231 날씨.csv', encoding='cp949')
display(weather_2024.head())

,지점,지점명,일시,기온(°C),강수량(mm),적설(cm)
0,108,서울,2023-01-01 01:00,1.5,NaN,NaN
1,108,서울,2023-01-01 02:00,1.5,NaN,NaN
2,108,서울,2023-01-01 03:00,1.6,NaN,NaN
3,108,서울,2023-01-01 04:00,1.5,NaN,NaN
4,108,서울,2023-01-01 05:00,0.8,NaN,NaN


,지점,지점명,일시,기온(°C),강수량(mm),적설(cm)
0,108,서울,2024-01-01 01:00,0.5,NaN,2.4
1,108,서울,2024-01-01 02:00,0.4,NaN,2.4
2,108,서울,2024-01-01 03:00,-0.1,NaN,2.4
3,108,서울,2024-01-01 04:00,-0.2,NaN,2.4
4,108,서울,2024-01-01 05:00,0.3,NaN,2.4


In [4]:
weather = pd.concat(
    [weather_2023, weather_2024],
    ignore_index=True
)

In [5]:
weather

,지점,지점명,일시,기온(°C),강수량(mm),적설(cm)
0,108,서울,2023-01-01 01:00,1.5,NaN,NaN
1,108,서울,2023-01-01 02:00,1.5,NaN,NaN
2,108,서울,2023-01-01 03:00,1.6,NaN,NaN
3,108,서울,2023-01-01 04:00,1.5,NaN,NaN
4,108,서울,2023-01-01 05:00,0.8,NaN,NaN
...,...,...,...,...,...,...
17539,108,서울,2024-12-31 20:00,0.0,NaN,NaN
17540,108,서울,2024-12-31 21:00,-0.7,NaN,NaN
17541,108,서울,2024-12-31 22:00,-1.0,NaN,NaN
17542,108,서울,2024-12-31 23:00,-1.4,NaN,NaN


In [6]:
weather.to_csv(
    'weather_2023-2024.csv',
    index=False,
    encoding='utf-8-sig'
)